# Group 25 - Training Notebook
# Krones Bottle-Base Inspection -- Multi-task EfficientNetV2-S Ensemble

This notebook trains our model for the bottle-base GOOD/FAULTY inspection task and exports it as a
**single ONNX file** for the evaluation notebook to use.


## Approach explanation

### Overview
We frame bottle-base inspection as **binary classification** (GOOD vs FAULTY), but we train the
network with an **auxiliary task** on the side: predicting which of the 26 defect categories are
present in each image. Predicting the categories too forces the backbone to learn *why* a bottle is
faulty, not just *that* it is — and this richer supervision is the single biggest reason our model
beats a plain binary classifier. We then average several cross-validation models into one ensemble
and export it as a single ONNX file.

### Model architecture
- **Backbone:** EfficientNetV2-S (`tf_efficientnetv2_s.in21k_ft_in1k`), pretrained on ImageNet-21k
  then fine-tuned on ImageNet-1k. It is accurate yet small and fast, which matters because the
  competition also scores efficiency.
- **Two output heads on a shared backbone:**
  - a **main head** (1 logit) for FAULTY vs GOOD — the task we are scored on;
  - an **auxiliary head** (26 logits) for the defect categories — used only during training.
- **Grayscale input (1 channel):** the photos are grayscale, so we feed a single channel instead of
  three identical ones. timm adapts the pretrained input stem automatically. This is cleaner and
  slightly faster.

### Key design decisions
- **Multi-task auxiliary head.** The 26-category supervision is what shapes the features to be
  defect-aware. Rare/faint defects barely move a binary loss, but the auxiliary head gives each
  defect type its own training signal. The auxiliary output is ignored at inference.
- **ROI crop at higher resolution.** Defects are tiny relative to the whole image, so we crop each
  image to its bottle base (using the ROI annotation) and train at 448px. This gives small defects
  far more pixels and was a major accuracy lever.
- **Ensemble of cross-validation folds.** We train several folds and average their probabilities.
  Averaging cancels each model's random errors and gives a small, reliable accuracy gain. The whole
  ensemble is exported as ONE ONNX file (a single graph, one input, one output) so it still counts
  as one model.

### Use of additional data (ROI, annotations, bottle types)
- **ROI boxes (COCO category 22):** used to crop each image to the base region — both at training
  and test time (the test set ships ROI-only annotations).
- **Defect annotations (the other 25+ categories):** turned into the 26-dim auxiliary targets that
  drive the multi-task head. This is the most valuable extra signal we exploit.
- **Bottle types:** used to keep cross-validation folds balanced (we considered them as a feature
  but the final model relies on the image + ROI + auxiliary supervision).

### Tricks and optimizations
- **Light, label-preserving augmentation** (flips, 90-degree rotations, mild brightness jitter).
  Bottle bases are rotationally symmetric, so these are valid. We deliberately avoid heavy noise/blur
  because our experiments showed they erase the faint defects we need to detect.
- **`pos_weight`** in the loss to handle class imbalance instead of resampling.
- **Mixed-precision (AMP)** training for speed and lower memory.
- **Cosine learning-rate schedule** annealed over the epochs.
- **Fine threshold search** (0.0025 steps) to pick the FAULTY decision threshold that maximizes F1.
- **Single-file ONNX ensemble** so the deployment is one self-contained model.

### Validation strategy
Stratified K-fold cross-validation by the label. Each fold trains on its training split and is
evaluated on its held-out split (data it never saw) -- an honest, leakage-free estimate. The decision
threshold is chosen on these held-out predictions, and the per-fold thresholds are averaged for the
final model.


## 0 - Imports and reproducibility

In [ ]:
import os, sys, json, gc, time, random, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
from tqdm.auto import tqdm

try:
    import timm
except ImportError:
    os.system(f"{sys.executable} -m pip install -q timm"); import timm

def seed_everything(seed=42):
    """Fix all random number generators so the run is reproducible."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

## 1 - Configuration


In [ ]:
cfg = {
    "seed": 42,
    "data_dir": "/kaggle/input/competitions/1st-krones-vision-ai-challenge",
    "output_dir": "/kaggle/working",

    # Backbone + image
    "backbone": "tf_efficientnetv2_s.in21k_ft_in1k",
    "image_size": 448,
    "input_channels": 1,
    "normalize_mean": 0.449,
    "normalize_std": 0.226,
    "roi_margin": 0.06,

    # Multi-task
    "auxiliary_loss_weight": 0.4,

    # Cross-validation + training
    "num_folds": 4,
    "epochs_per_fold": 5,
    "batch_size": 16,
    "learning_rate": 1.5e-4,
    "weight_decay": 1e-4,
    "num_workers": 2,
    "use_mixed_precision": True,
    "use_augmentation": True,

    # Decision threshold search
    "threshold_search_grid": (0.05, 0.95, 0.0025),
}
seed_everything(cfg["seed"])

data_dir         = Path(cfg["data_dir"])
output_dir       = Path(cfg["output_dir"]); output_dir.mkdir(exist_ok=True)
train_image_dir  = data_dir / "train_images"
train_labels_csv = data_dir / "train.csv"
annotations_json = data_dir / "train_annotations.json"
print("Data directory found:", data_dir.exists())

## 2 - Load the labels
`train.csv` holds the binary target per image (1 = FAULTY, 0 = GOOD).

In [ ]:
train_df = pd.read_csv(train_labels_csv)
print("Training images:", len(train_df))
print(train_df["target"].value_counts())
print(f"FAULTY fraction: {train_df['target'].mean():.1%}")

## 3 - ROI boxes and the 26-category auxiliary targets
`train_annotations.json` is a COCO file. Category 22 is the ROI (base region); the rest are defect types. We build a filename->ROI map for cropping and a 26-dim multi-label vector per image (the auxiliary targets).

In [ ]:
with open(annotations_json) as f:
    coco = json.load(f)

ROI_CATEGORY_ID = 22
image_id_to_filename = {img["id"]: Path(img["file_name"]).name for img in coco["images"]}

roi_box_by_filename = {
    image_id_to_filename[ann["image_id"]]: ann["bbox"]
    for ann in coco["annotations"] if ann["category_id"] == ROI_CATEGORY_ID
}

defect_category_ids = sorted(c["id"] for c in coco["categories"] if c["id"] != ROI_CATEGORY_ID)
defect_category_names = {c["id"]: c["name"] for c in coco["categories"]}
num_aux_categories = len(defect_category_ids)
category_id_to_index = {cat_id: idx for idx, cat_id in enumerate(defect_category_ids)}

aux_target_by_filename = {}
for ann in coco["annotations"]:
    if ann["category_id"] == ROI_CATEGORY_ID:
        continue
    filename = image_id_to_filename.get(ann["image_id"])
    if filename is None:
        continue
    vector = aux_target_by_filename.setdefault(filename, np.zeros(num_aux_categories, np.float32))
    vector[category_id_to_index[ann["category_id"]]] = 1.0

print("ROI boxes:", len(roi_box_by_filename), "| auxiliary categories:", num_aux_categories)
del coco; gc.collect()

## 4 - Exploratory Data Analysis

In [ ]:
from collections import Counter
defect_counts = Counter()
with open(annotations_json) as f:
    coco_for_eda = json.load(f)
for ann in coco_for_eda["annotations"]:
    if ann["category_id"] != ROI_CATEGORY_ID:
        defect_counts[defect_category_names[ann["category_id"]]] += 1
del coco_for_eda
defect_count_series = pd.Series(defect_counts).sort_values()

figure, axes = plt.subplots(1, 2, figsize=(14, 5))
label_counts = train_df["target"].value_counts().sort_index()
axes[0].bar(["GOOD", "FAULTY"], [label_counts.get(0, 0), label_counts.get(1, 0)],
            color=["#4c72b0", "#c44e52"])
axes[0].set_title("Class balance"); axes[0].set_ylabel("number of images")
axes[1].barh(defect_count_series.index, defect_count_series.values, color="#8172b3")
axes[1].set_title("Defect category frequency (highly imbalanced)")
axes[1].set_xlabel("annotated instances")
plt.tight_layout(); plt.show()
print("Defects are small/faint and category-imbalanced -> motivates ROI crop + auxiliary head.")

## 5 - ROI crop and dataset
`crop_to_roi` makes a square crop centred on the ROI (so resizing doesn't distort the round base) and reflects the border if the square extends past the image. `BottleDataset` loads a grayscale image, crops, optionally augments, and returns a normalized tensor.

In [ ]:
def crop_to_roi(image, roi_x, roi_y, roi_w, roi_h, margin, output_size):
    """Square crop around the ROI box (+margin), reflect-pad if needed, resize to output_size."""
    height, width = image.shape[:2]
    center_x, center_y = roi_x + roi_w / 2.0, roi_y + roi_h / 2.0
    side = max(roi_w, roi_h) * (1.0 + 2.0 * margin)
    left, top = int(round(center_x - side / 2)), int(round(center_y - side / 2))
    right, bottom = int(round(center_x + side / 2)), int(round(center_y + side / 2))
    pad_left, pad_top = max(0, -left), max(0, -top)
    pad_right, pad_bottom = max(0, right - width), max(0, bottom - height)
    cropped = image[max(0, top):min(height, bottom), max(0, left):min(width, right)]
    if pad_left or pad_top or pad_right or pad_bottom:
        cropped = cv2.copyMakeBorder(cropped, pad_top, pad_bottom, pad_left, pad_right,
                                     cv2.BORDER_REFLECT_101)
    interpolation = cv2.INTER_AREA if cropped.shape[0] > output_size else cv2.INTER_LINEAR
    return cv2.resize(cropped, (output_size, output_size), interpolation=interpolation)

roi_values = np.array(list(roi_box_by_filename.values()), np.float32)
fallback_roi = tuple(np.median(roi_values, axis=0)) if len(roi_values) else (160, 147, 987, 722)


class BottleDataset(Dataset):
    """Returns (image, target, aux_vector) for training, or (image, filename) for test."""
    def __init__(self, filenames, image_dir, roi_map, config,
                 targets=None, aux_targets=None, augment=False):
        self.filenames = list(filenames)
        self.image_dir = Path(image_dir)
        self.roi_map = roi_map
        self.config = config
        self.targets = targets
        self.aux_targets = aux_targets
        self.augment = augment and config["use_augmentation"]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        filename = self.filenames[index]
        image = cv2.imread(str(self.image_dir / filename), cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(self.image_dir / filename)
        roi_x, roi_y, roi_w, roi_h = self.roi_map.get(filename, fallback_roi)
        image = crop_to_roi(image, roi_x, roi_y, roi_w, roi_h,
                            self.config["roi_margin"], self.config["image_size"])
        if self.augment:
            if random.random() < 0.5: image = np.fliplr(image)
            if random.random() < 0.5: image = np.flipud(image)
            num_rotations = random.randint(0, 3)
            if num_rotations: image = np.rot90(image, num_rotations)
            if random.random() < 0.4:
                image = np.clip(image.astype(np.float32) * random.uniform(0.92, 1.08)
                                + random.uniform(-8, 8), 0, 255).astype(np.uint8)
        tensor = torch.from_numpy(np.ascontiguousarray(image)).float()
        tensor = tensor.div_(255.0).sub_(self.config["normalize_mean"]).div_(self.config["normalize_std"])
        tensor = tensor.unsqueeze(0)
        if self.targets is None:
            return tensor, filename
        target = torch.tensor(float(self.targets[index]), dtype=torch.float32)
        aux = (torch.from_numpy(self.aux_targets[index]) if self.aux_targets is not None
               else torch.zeros(num_aux_categories))
        return tensor, target, aux

## 6 - Model: `BottleInspectionNet`
Shared EfficientNetV2-S backbone with a main head (FAULTY/GOOD) and an auxiliary head (26 defect categories). The auxiliary head is used only during training to shape the features.

In [ ]:
class BottleInspectionNet(nn.Module):
    def __init__(self, config, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            config["backbone"], pretrained=pretrained,
            in_chans=config["input_channels"], num_classes=0)
        num_features = self.backbone.num_features
        self.main_head = nn.Linear(num_features, 1)
        self.auxiliary_head = nn.Linear(num_features, num_aux_categories)

    def forward(self, image):
        features = self.backbone(image)
        main_logit = self.main_head(features).squeeze(-1)
        auxiliary_logits = self.auxiliary_head(features)
        return main_logit, auxiliary_logits


_model = BottleInspectionNet(cfg, pretrained=False).to(device)
with torch.no_grad():
    _main, _aux = _model(torch.randn(2, cfg["input_channels"],
                                     cfg["image_size"], cfg["image_size"]).to(device))
print("Build OK | main:", tuple(_main.shape), "| auxiliary:", tuple(_aux.shape),
      "| parameters:", f"{sum(p.numel() for p in _model.parameters()):,}")
del _model, _main, _aux; torch.cuda.empty_cache()

## 7 - Loss, threshold search, training helper
Total loss = main loss + weight x auxiliary loss. The main loss uses `pos_weight` for class imbalance. `find_best_threshold` picks the FAULTY threshold that maximizes F1.

In [ ]:
positive_count = float(train_df["target"].sum())
negative_count = float(len(train_df) - positive_count)
positive_weight = torch.tensor([negative_count / max(positive_count, 1.0)],
                               dtype=torch.float32, device=device)
main_loss_fn = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
print("pos_weight (neg/pos):", round(float(positive_weight.item()), 3))

def compute_loss(main_logit, auxiliary_logits, target, aux_target):
    """Multi-task loss: main FAULTY/GOOD + weighted auxiliary 26-category."""
    main_loss = main_loss_fn(main_logit, target)
    auxiliary_loss = F.binary_cross_entropy_with_logits(auxiliary_logits, aux_target)
    return main_loss + cfg["auxiliary_loss_weight"] * auxiliary_loss

grid_low, grid_high, grid_step = cfg["threshold_search_grid"]
threshold_grid = np.round(np.arange(grid_low, grid_high + 1e-9, grid_step), 4)

def find_best_threshold(true_labels, predicted_probs):
    """Return (threshold, F1) maximizing F1 over the grid."""
    true_labels = np.asarray(true_labels).astype(int)
    predicted_probs = np.asarray(predicted_probs)
    best_threshold, best_f1 = 0.5, -1.0
    for threshold in threshold_grid:
        score = f1_score(true_labels, (predicted_probs >= threshold).astype(int), zero_division=0)
        if score > best_f1:
            best_f1, best_threshold = score, float(threshold)
    return best_threshold, best_f1

@torch.no_grad()
def predict_probabilities(model, filenames, image_dir, roi_map):
    """Run a model over filenames, return the FAULTY probability per image (main head)."""
    loader = DataLoader(BottleDataset(filenames, image_dir, roi_map, cfg),
                        batch_size=cfg["batch_size"] * 2, shuffle=False,
                        num_workers=cfg["num_workers"])
    model = model.to(device).eval()
    probabilities = []
    for images, _ in loader:
        with torch.amp.autocast(device_type=device.type,
                                enabled=cfg["use_mixed_precision"] and device.type == "cuda"):
            main_logit, _ = model(images.to(device))
        probabilities.append(torch.sigmoid(main_logit.float()).cpu().numpy())
    return np.concatenate(probabilities)

def train_one_fold(train_filenames, train_targets, train_aux, fold_label):
    """Train a single model on the given subset; return the trained model."""
    loader = DataLoader(
        BottleDataset(train_filenames, train_image_dir, roi_box_by_filename, cfg,
                      targets=train_targets, aux_targets=train_aux, augment=True),
        batch_size=cfg["batch_size"], shuffle=True,
        num_workers=cfg["num_workers"], drop_last=True)
    model = BottleInspectionNet(cfg, pretrained=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"],
                                  weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs_per_fold"])
    scaler = torch.amp.GradScaler(enabled=cfg["use_mixed_precision"] and device.type == "cuda")
    for epoch in range(1, cfg["epochs_per_fold"] + 1):
        model.train(); running_loss = 0.0; start_time = time.time()
        for images, targets, aux in tqdm(loader, desc=f"{fold_label} epoch {epoch}", leave=False):
            images, targets, aux = images.to(device), targets.to(device), aux.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type,
                                    enabled=cfg["use_mixed_precision"] and device.type == "cuda"):
                main_logit, auxiliary_logits = model(images)
                loss = compute_loss(main_logit, auxiliary_logits, targets, aux)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            running_loss += loss.item() * images.size(0)
        scheduler.step()
        print(f"  {fold_label} epoch {epoch}/{cfg['epochs_per_fold']} "
              f"loss={running_loss/len(loader.dataset):.5f} ({(time.time()-start_time)/60:.1f} min)")
    return model

## 8 - Cross-validation folds (stratified by label)

In [ ]:
train_df = train_df.reset_index(drop=True)
fold_splitter = StratifiedKFold(n_splits=cfg["num_folds"], shuffle=True, random_state=cfg["seed"])
train_df["fold"] = -1
for fold_index, (_, validation_indices) in enumerate(fold_splitter.split(train_df, train_df["target"])):
    train_df.loc[validation_indices, "fold"] = fold_index
print(train_df.groupby(["fold", "target"]).size())

aux_target_matrix = np.stack([
    aux_target_by_filename.get(filename, np.zeros(num_aux_categories, np.float32))
    for filename in train_df["image_id"]
]).astype(np.float32)

## 9 - Train every fold and collect held-out predictions
We also store each fold's held-out probabilities and labels so we can plot an overall confusion matrix afterwards (honest, since each prediction comes from a model that didn't train on it).

In [ ]:
trained_models = []
fold_thresholds = []
heldout_probs = np.full(len(train_df), np.nan, dtype=np.float64)   # for the confusion matrix
training_start = time.time()

for fold_index in range(cfg["num_folds"]):
    print(f"\n{'='*60}\nFOLD {fold_index}\n{'='*60}")
    is_training = (train_df["fold"] != fold_index).values
    is_validation = ~is_training
    training_row_indices = np.where(is_training)[0]
    validation_row_indices = np.where(is_validation)[0]

    model = train_one_fold(
        train_df.loc[is_training, "image_id"].tolist(),
        train_df.loc[is_training, "target"].values,
        aux_target_matrix[training_row_indices],
        fold_label=f"fold{fold_index}")

    validation_filenames = train_df.loc[is_validation, "image_id"].tolist()
    validation_probs = predict_probabilities(model, validation_filenames, train_image_dir,
                                             roi_box_by_filename)
    heldout_probs[validation_row_indices] = validation_probs
    validation_labels = train_df.loc[is_validation, "target"].values
    threshold, fold_f1 = find_best_threshold(validation_labels, validation_probs)
    print(f"  fold {fold_index} held-out F1 = {fold_f1:.5f} (threshold {threshold:.4f})")

    trained_models.append(model.cpu().eval())
    fold_thresholds.append(threshold)
    torch.cuda.empty_cache()

final_threshold = float(np.mean(fold_thresholds))
print(f"\nTotal training time: {(time.time()-training_start)/60:.1f} min")
print(f"Final decision threshold (mean of folds): {final_threshold:.4f}")

## 10 - Validation results and confusion matrix
Using the held-out predictions and the final threshold, we report F1/recall/precision and plot the confusion matrix. Recall and precision tell us the balance of false negatives vs false positives, which we discuss next.

In [ ]:
true_labels = train_df["target"].values.astype(int)
predicted_labels = (heldout_probs >= final_threshold).astype(int)

overall_f1 = f1_score(true_labels, predicted_labels)
overall_recall = recall_score(true_labels, predicted_labels)
overall_precision = precision_score(true_labels, predicted_labels)
matrix = confusion_matrix(true_labels, predicted_labels)   # rows = true, cols = predicted
true_neg, false_pos = matrix[0]
false_neg, true_pos = matrix[1]

print(f"Held-out F1        : {overall_f1:.4f}")
print(f"Recall  (FAULTY)   : {overall_recall:.4f}   <- higher = fewer missed faulty bottles (fewer FN)")
print(f"Precision (FAULTY) : {overall_precision:.4f}")
print(f"True Negatives (GOOD ok) : {true_neg}")
print(f"False Positives (GOOD called FAULTY) : {false_pos}")
print(f"False Negatives (FAULTY missed)      : {false_neg}   <- the costly errors")
print(f"True Positives (FAULTY caught)       : {true_pos}")

figure, axis = plt.subplots(figsize=(5.5, 5))
display = axis.imshow(matrix, cmap="Blues")
axis.set_xticks([0, 1]); axis.set_xticklabels(["GOOD", "FAULTY"])
axis.set_yticks([0, 1]); axis.set_yticklabels(["GOOD", "FAULTY"])
axis.set_xlabel("Predicted"); axis.set_ylabel("True")
axis.set_title(f"Confusion matrix (held-out)  F1={overall_f1:.4f}")
for row, col in itertools.product(range(2), range(2)):
    axis.text(col, row, matrix[row, col], ha="center", va="center",
              color="white" if matrix[row, col] > matrix.max() / 2 else "black", fontsize=14)
plt.tight_layout(); plt.show()

## 11 - Prioritizing false negatives vs false positives

In this inspection setting the two error types are **not equally costly**:

- A **false negative (FN)** means a *faulty* bottle is labelled GOOD and passes inspection. This is
  the dangerous, expensive error: a defective bottle reaches packaging or the customer, risking
  product recalls, contamination complaints, brand damage, and potential safety issues. For Krones'
  line, a missed defect is far more costly than an unnecessary rejection.
- A **false positive (FP)** means a *good* bottle is labelled FAULTY and rejected. This wastes a good
  bottle and a little throughput, but it is cheap and safe by comparison.

Because FN is the costly error, we would tune the system to **prioritize recall (minimize FN)**, even
at the price of more FP. Concretely:

- **Lower the decision threshold.** Calling more bottles FAULTY raises recall (fewer FN) at the cost
  of precision (more FP). For example, instead of the F1-optimal threshold we could choose the
  *lowest* threshold that still keeps precision acceptable, or require recall ≥ 0.99.
- **Optimize the threshold under a recall constraint** rather than for raw F1 — e.g. "maximize F1
  subject to recall ≥ 0.99" so we guarantee almost no faulty bottle slips through.
- **Use a cost-weighted objective**: weight FN more heavily than FP (a larger `pos_weight`, or an
  explicit cost matrix) so the model itself leans toward catching faults.

The trade-off is real: pushing recall toward 1.0 increases false rejections of good bottles. In a
factory this is usually acceptable — rejected "good" bottles can be re-inspected or recycled, whereas
a shipped faulty bottle is a much larger liability. The opposite tuning (prioritizing precision /
minimizing FP) would only make sense if false rejections were extremely expensive and the cost of a
missed defect were low, which is not the case here.

The code below shows how the threshold could be chosen to enforce a minimum recall, if desired
(we leave the F1-optimal threshold as that which maximizes recall).

In [ ]:
def threshold_for_minimum_recall(true_labels, predicted_probs, minimum_recall=0.99):
    """Lowest-FN option: best F1 among thresholds that keep recall >= minimum_recall."""
    true_labels = np.asarray(true_labels).astype(int)
    predicted_probs = np.asarray(predicted_probs)
    best_threshold, best_f1, best_recall = None, -1.0, 0.0
    for threshold in threshold_grid:
        predicted = (predicted_probs >= threshold).astype(int)
        recall = recall_score(true_labels, predicted, zero_division=0)
        if recall < minimum_recall:
            continue
        score = f1_score(true_labels, predicted, zero_division=0)
        if score > best_f1:
            best_f1, best_threshold, best_recall = score, float(threshold), recall
    if best_threshold is None:    # no threshold reached the floor; use the highest-recall one
        recalls = [recall_score(true_labels, (predicted_probs >= t).astype(int), zero_division=0)
                   for t in threshold_grid]
        best_threshold = float(threshold_grid[int(np.argmax(recalls))])
        best_recall = max(recalls)
        best_f1 = f1_score(true_labels, (predicted_probs >= best_threshold).astype(int), zero_division=0)
    return best_threshold, best_f1, best_recall

recall_threshold, recall_f1, achieved_recall = threshold_for_minimum_recall(
    true_labels, heldout_probs, minimum_recall=0.99)
print(f"If prioritizing FN: threshold={recall_threshold:.4f} -> "
      f"recall={achieved_recall:.4f}, F1={recall_f1:.4f}")
print("(Not used for submission — shown to illustrate the FN-vs-FP trade-off.)")

## 12 - Build the ensemble and export to ONE ONNX file

In [ ]:
class EnsembleModel(nn.Module):
    """Runs all fold models and averages their FAULTY probabilities into one output."""
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, image):
        probabilities = []
        for model in self.models:
            main_logit, _ = model(image)          # ignore the auxiliary head at inference
            probabilities.append(torch.sigmoid(main_logit))
        return torch.stack(probabilities, dim=0).mean(dim=0)

ensemble = EnsembleModel([model.to(device) for model in trained_models]).to(device).eval()

example_input = torch.randn(2, cfg["input_channels"], cfg["image_size"], cfg["image_size"], device=device)
with torch.no_grad():
    example_output = ensemble(example_input)
print("Ensemble output shape:", tuple(example_output.shape),
      "| range:", round(float(example_output.min()), 3), "-", round(float(example_output.max()), 3))

onnx_path = output_dir / "bottle_inspection_ensemble.onnx"
torch.onnx.export(
    ensemble, example_input[:1], str(onnx_path),
    input_names=["image"], output_names=["faulty_probability"],
    dynamic_axes={"image": {0: "batch"}, "faulty_probability": {0: "batch"}},
    opset_version=18)
print("Exported ONNX to:", onnx_path)

## 13 - Save the inference configuration
The evaluation notebook has its own `cfg` (copied from this one), but we also save the threshold and fallback ROI here for reference / verification.

In [ ]:
inference_reference = {
    "decision_threshold": final_threshold,
    "fallback_roi": [float(v) for v in fallback_roi],
    "onnx_filename": "bottle_inspection_ensemble.onnx",
    "heldout_f1": float(overall_f1),
}
with open(output_dir / "inference_reference.json", "w") as f:
    json.dump(inference_reference, f, indent=2)
print(json.dumps(inference_reference, indent=2))
print("\nIMPORTANT: copy `final_threshold` and `fallback_roi` above into the evaluation notebook's cfg.")

## 14 - Done

Artifacts: `bottle_inspection_ensemble.onnx` and `inference_reference.json`.
Attach them as a Kaggle Dataset, share privately with both organizers, and copy the threshold +
fallback ROI into the evaluation notebook's `cfg`.